#### What is the Silver Layer?

The Silver layer takes **raw messy data from Bronze** and makes it **clean, consistent, and ready for analysis**.

Think of it like editing a rough draft - you fix spelling, remove junk, and organize everything properly.

---

#### What we do in every Silver notebook:

| Step | What it does | Why |
| --- | --- | --- |
| 1. Read | Pull data from Bronze table | Starting point |
| 2. Drop | Remove columns we don't need (e.g., `url`) | Keep only useful data |
| 3. Rename | Change column names to snake_case | Consistency |
| 4. Filter Nulls | Remove rows with missing key values | Data quality |
| 5. Deduplicate | Remove duplicate rows | Avoid counting things twice |
| 6. Format | Apply title case to text columns | Clean presentation |
| 7. Write | Save to Silver Delta table | Ready for analysis |

---
#### Entity Relationship Diagram - Silver Layer (`formula1.silver`)

This diagram shows all 6 tables in the silver schema, their columns, and how they relate to each other through foreign keys.

**Figma Source:** [Formula1 DataWarehouse ERD](https://www.figma.com/design/qwT3ax5c5f804QC3Dqjnbt/Formula1_DataWarehouse_ERD?node-id=0-1&p=f&m=draw)

#### Relationships Summary:

| Relationship | Type | Description |
| --- | --- | --- |
| `races.circuit_id` -> `circuits.circuit_id` | Many-to-One | Each race takes place at one circuit |
| `results.driver_id` -> `drivers.driver_id` | Many-to-One | Each result belongs to one driver |
| `results.constructor_id` -> `constructors.constructor_id` | Many-to-One | Each result belongs to one team |
| `sprints.driver_id` -> `drivers.driver_id` | Many-to-One | Each sprint result belongs to one driver |
| `sprints.constructor_id` -> `constructors.constructor_id` | Many-to-One | Each sprint result belongs to one team |
| `results` <-> `races` | Implicit | Linked via season, round, race_name, race_date columns |
| `sprints` <-> `races` | Implicit | Linked via season, round, race_name, race_date columns |

**Note:** `results` and `sprints` don't have a direct `circuit_id` FK - they embed race info (season, round, race_name, race_date) instead, which can be joined to the `races` table.

In [0]:
displayHTML('<iframe style="border: 1px solid rgba(0, 0, 0, 0.1);" width="800" height="450" src="https://embed.figma.com/design/qwT3ax5c5f804QC3Dqjnbt/Formula1_DataWarehouse_ERD?node-id=0-1&embed-host=share" allowfullscreen></iframe>')

---
#### Code Flow - Syntax Reference

Below is the general pattern used in all Silver transformation notebooks.

##### 1. Load Configuration
```python
%run ../00-common/01.environment-config
```
Imports shared variables like `catalog_name`, `bronze_schema`, `silver_schema`.

##### 2. Set Variables
```python
bronze_table = f'{catalog_name}.{bronze_schema}.circuits'
silver_table = f'{catalog_name}.{silver_schema}.circuits'
```
Define source (Bronze) and target (Silver) table names.

##### 3. Import Functions
```python
from pyspark.sql.functions import *
```
Imports all Spark SQL functions (col, initcap, concat_ws, etc.).

##### 4. Read Bronze Table
```python
df = spark.read.table(bronze_table)
```
Reads the raw data from a Delta table into a DataFrame.

##### 5. Drop Columns
```python
df_drop = df.drop('url')
```
Removes columns that are not needed for analysis.

##### 6. Rename Columns
```python
# Single column
df_renamed = df_drop.withColumnRenamed('old_name', 'new_name')

# Multiple columns at once
df_renamed = df_drop.withColumnsRenamed({
    'circuitId': 'circuit_id',
    'raceName': 'race_name'
})
```
Renames columns to snake_case for consistency.

##### 7. Filter Null Values
```python
df_valid = df_renamed.filter(col('id').isNotNull())

# Multiple conditions
df_valid = df_renamed.filter(
    col('season').isNotNull() &
    col('driver_id').isNotNull()
)
```
Removes rows where key columns have missing values.

##### 8. Remove Duplicates
```python
# Remove exact duplicates across all columns
df_distinct = df_valid.distinct()

# Remove duplicates based on specific columns
df_distinct = df_valid.dropDuplicates(['circuit_id'])
```
Ensures each record appears only once.

##### 9. Format Text (Title Case)
```python
df_final = df_distinct.withColumn('name', initcap(col('name')))
```
`initcap()` capitalizes the first letter of each word (e.g., "british" becomes "British").

##### 10. Concatenate Columns (Drivers only)
```python
df_concat = df.withColumn('driver_name',
    concat_ws(' ', col('name.givenName'), col('name.familyName')))
```
`concat_ws(' ', ...)` joins multiple values with a space in between. Used for nested JSON fields.

##### 11. Write to Silver Table
```python
(
    df_final.write
    .mode('overwrite')
    .format('delta')
    .saveAsTable(silver_table)
)
```
Saves the cleaned data to the Silver Delta table, replacing existing data.

---

#### Notebooks in this folder:

| Notebook | Data | Key transformations |
| --- | --- | --- |
| **`01) Transformation Circuits data`** | Track details | Drop url, rename lat/long, deduplicate, title case |
| **`02) Transformation Races data`** | Race schedule | Drop url, rename raceName, deduplicate, title case |
| **`03) Transformation Conductor data`** | Teams | Drop url, rename constructorId/name, title case nationality |
| **`04) Transformation Drivers data`** | Driver info | Rename, concatenate nested name fields, title case |
| **`05) Transform Results Data`** | Race results | Rename 9 columns, filter nulls, deduplicate, title case |
| **`06) Transform Sprints Data`** | Sprint results | Same as Results but for sprint races |